## Analyze model predictions using PCA, tSNE, and UMAP
---
Compares three representations of the same specimens side by side: the 28 sparse landmarks, the
dense (population) correspondences, and the NSM latent codes. All three come from the same
`all_vtk_files` order, so one `specimens` table (species / family / trait / color / marker, joined
from `lizard_species_list.csv`) drives every plot.

*Last edited 10 Sep 2026 by K. Wolcott*

In [ ]:
# Imports, paths, and config

import os, re, json, ast, torch
import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap.umap_ as umap

from NSM.plotting import (load_mrk_json, plot_life_history_legend, plot_family_color_legend,
                          sort_key, plotly_color)
from NSM.morphometrics import *   # gm_prcomp, two_d_array, mshape, centroid_size, ...

# ---------------------------------------------------------------- TO DO: edit
RUN          = "run_v72"                      # training attempt directory
ATLAS_RUN    = "2026_07-15_13_06_22/"         # atlas/builder run that produced the landmark sets
ATLAS_ROOT = Path("final_dataset_aug26/atlas/")
CKPT         = "2500"                         # latent code checkpoint to analyze
# -----------------------------------------------------------------------------

cwd       = Path.cwd()
base_wd   = cwd.parent
train_dir = base_wd / RUN
os.chdir(train_dir)
print(f"Working directory: {os.getcwd()}")

ATLAS_DIR       = ATLAS_ROOT / ATLAS_RUN / "atlas"
SPARSE_LM_DIR   = ATLAS_ROOT / ATLAS_RUN / "alignedLMs"
DENSE_LM_DIR    = ATLAS_ROOT / ATLAS_RUN / "population_correspondences"
SPARSE_MEAN_FN  = ATLAS_DIR / "atlas_sparse_landmarks.mrk.json"
DENSE_MEAN_FN   = ATLAS_DIR / "atlas_dense_correspondences.mrk.json"
MEAN_MESH_FN    = ATLAS_DIR / "atlas_model.ply"

OUT_DIR = Path("pca_tsne_umap_results")
OUT_DIR.mkdir(exist_ok=True)
print(f"Outputs will be written to: {OUT_DIR.resolve()}")

for p in [ATLAS_DIR, SPARSE_LM_DIR, DENSE_LM_DIR, SPARSE_MEAN_FN, DENSE_MEAN_FN, MEAN_MESH_FN]:
    print(("  OK   " if p.exists() else "  MISS ") + str(p))

# Load config and filenames
with open("model_params_config.json") as f:
    cfg = json.load(f)
print(f"\033[92mLoaded config from model_params_config.json\033[0m")

all_vtk_files = [os.path.basename(f) for f in cfg["list_mesh_paths"]]
print(f"{len(all_vtk_files)} meshes listed in config")

pat = re.compile(r"^(?P<species>[\w\s\-]+)[\-_ ]+[\w\d]+[\-_ ]+(?P<vertebra>[CTL]?\d+)", re.IGNORECASE)

In [ ]:
# Load NSM latent codes

CKPT_PATH = f"latent_codes/{CKPT}.pth"
latent_ckpt = torch.load(CKPT_PATH, map_location="cpu")
codes = latent_ckpt["latent_codes"]["weight"].detach().cpu().numpy()
print(f"Latent codes: {codes.shape}")
assert len(codes) == len(all_vtk_files), (
    f"{len(codes)} latent codes but {len(all_vtk_files)} meshes in config -- order/count mismatch")

### Specimen metadata (species / family / trait / color / marker)

In [ ]:
# Build specimen metadata table -- one row per mesh, in the same order as `codes`

SPECIES_CSV = "../lizard_species_list.csv"

parsed = [pat.match(f) for f in all_vtk_files]
specimens = pd.DataFrame({"mesh":        all_vtk_files,
                          "specimen_id": [m.group("species") if m else None for m in parsed],
                          "vertebra":    [m.group("vertebra").upper() if m else None for m in parsed]})
print(f"Parsed {specimens['specimen_id'].notna().sum()} / {len(specimens)} filenames")

sdf = pd.read_csv(SPECIES_CSV)
sdf["marker"] = sdf["marker"].astype(str).str.strip().str.strip("'" + chr(34))
sdf["color"]  = sdf["color"].apply(ast.literal_eval)
specimens = specimens.merge(sdf, left_on="specimen_id", right_on="specimen", how="left")

unmatched = specimens.loc[specimens["family"].isna(), "specimen_id"].dropna().unique()
print(f"{specimens['family'].notna().sum()} / {len(specimens)} specimens matched to {SPECIES_CSV}")
if len(unmatched):
    print(f"\033[33mUnmatched specimen IDs ({len(unmatched)}):\033[0m {sorted(unmatched)[:10]}")

REGION_NAMES = {"C": "CERVICAL", "T": "THORACIC", "L": "LUMBAR"}
specimens["region"] = specimens["vertebra"].str[0].map(REGION_NAMES)

print("Specimens dataframe head:\n", specimens.head())

# One color per broad clade (matches plot_family_color_legend elsewhere) -- computed once and
# reused both for the inline PNG legends below and the standalone legend cell at the end.
family_base_colors = (specimens.drop_duplicates("broad_taxon_for_plotting")
                     .set_index("broad_taxon_for_plotting")["color"].to_dict())

### Load the three shape representations
Sparse landmarks and dense correspondences are both GPA-aligned/scaled already, so `gm_prcomp`
(Procrustes PCA, from `NSM.morphometrics`) runs directly on them. Latent codes get plain `sklearn`
PCA since they aren't Procrustes shape coordinates.

In [ ]:
# 28 sparse landmarks

sparse_coords = np.stack([load_mrk_json(SPARSE_LM_DIR / (os.path.splitext(f)[0] + ".mrk.json"))[0]
                          for f in all_vtk_files])
sparse_mean, _ = load_mrk_json(SPARSE_MEAN_FN)
print(f"Sparse landmarks: {sparse_coords.shape}")
assert sparse_mean.shape == sparse_coords.shape[1:], "sparse atlas / specimen landmark count mismatch"

pca_sparse = gm_prcomp(sparse_coords)
print(f"{pca_sparse['x'].shape[1]} non-trivial PCs from {sparse_coords.shape[1]*3} sparse coordinates")

In [ ]:
# Dense correspondences -- gm_prcomp here costs roughly 30s at ~5000 points x 3 dims x this many
# specimens; the full-SVD PCA runs once and every downstream tSNE/UMAP call reuses `pca_dense["x"]`.

dense_coords = np.stack([load_mrk_json(DENSE_LM_DIR / (os.path.splitext(f)[0] + ".mrk.json"))[0]
                         for f in all_vtk_files])
dense_mean, _ = load_mrk_json(DENSE_MEAN_FN)
print(f"Dense correspondences: {dense_coords.shape}")
assert dense_mean.shape == dense_coords.shape[1:], "dense atlas / specimen point count mismatch"

pca_dense = gm_prcomp(dense_coords)
print(f"{pca_dense['x'].shape[1]} non-trivial PCs from {dense_coords.shape[1]*3} dense coordinates")

In [ ]:
# NSM latent codes

pca_latent_model = PCA(n_components=5)
latent_scores = pca_latent_model.fit_transform(codes)
latent_prop = pca_latent_model.explained_variance_ratio_
print(f"Latent PCA: {latent_scores.shape[1]} components, "
      f"PC1+PC2 = {100*(latent_prop[0]+latent_prop[1]):.1f}% of variance")

### PCA

In [ ]:
# Shared helper: one PC-pair scatter, one subplot per dataset

def _family_legend_traces():
    """One legend-only swatch per broad clade, for the static PNG export (see pc_pair_grid)."""
    return [go.Scatter(x=[None], y=[None], mode="markers",
                       marker=dict(size=20, color=plotly_color(col), symbol="circle"),
                       name=fam.upper(), legendgroup=fam, showlegend=True)
           for fam, col in family_base_colors.items() if isinstance(fam, str)]


def pc_pair_grid(datasets, pc_i, pc_j, title, axis_prefix="PC", outstem=None, width=2100, height=800, show_legend=True):
    """datasets: list of (panel_title, scores_2d_or_more, prop_variance_array_or_None)."""
    fig = make_subplots(rows=1, cols=len(datasets), subplot_titles=[d[0] for d in datasets],
                        horizontal_spacing=0.06)

    for col, (panel_title, scores, prop) in enumerate(datasets, start=1):
        groups = defaultdict(list)
        for idx, row in specimens.iterrows():
            groups[row["specimen_id"]].append((row["vertebra"], scores[idx, pc_i], scores[idx, pc_j],
                                               row["color"]))
        for name, points in groups.items():
            points_sorted = sorted(points, key=sort_key)
            vlabs, xv, yv, colors = zip(*points_sorted)
            fig.add_trace(go.Scatter(x=xv, y=yv, mode="markers", name=name,
                                     legendgroup=name, showlegend=(col == 1),
                                     marker=dict(color=plotly_color(colors[0]), size=6, symbol="circle"),
                                     text=vlabs,
                                     hovertemplate=f"Specimen: {name}<br>Vertebra: %{{text}}<extra></extra>"),
                          row=1, col=col)

        xlab = f"{axis_prefix}{pc_i+1}: {100*prop[pc_i]:.2f}%" if prop is not None else f"{axis_prefix}{pc_i+1}"
        ylab = f"{axis_prefix}{pc_j+1}: {100*prop[pc_j]:.2f}%" if prop is not None else f"{axis_prefix}{pc_j+1}"
        fig.update_xaxes(title_text=xlab, row=1, col=col)
        anchor = f"x{col}" if col > 1 else "x"
        fig.update_yaxes(title_text=ylab, row=1, col=col, scaleanchor=anchor, scaleratio=1)

    fig.update_layout(width=width, height=height, title=title, plot_bgcolor="white",
                      legend=dict(x=1.02, y=0.5, xanchor="left", yanchor="middle"))

    if outstem:
        fig.write_html(str(OUT_DIR / f"{outstem}.html"), include_plotlyjs="cdn")  # per-specimen legend, untouched

        fig_png = go.Figure(fig)                # separate copy -- HTML stays exactly as built above
        for tr in fig_png.data:
            tr.showlegend = False                # drop the ~150-entry per-specimen legend
            if tr.marker.symbol == "circle" and tr.x[0] is not None:   # real data traces, not legend swatches
                tr.marker.size = 12                                     # was 6 (set in the shared loop above)     

        for ann in fig_png.layout.annotations:    # blank per-panel titles -- added once after stitching instead
            ann.text = ""

        fig_png.update_layout(font=dict(size=50))
        fig_png.update_layout(title=None, margin=dict(t=40, b=150))
        fig_png.update_xaxes(title_font=dict(size=50))
        fig_png.update_yaxes(title_font=dict(size=50))
        fig_png.update_xaxes(showticklabels=False, ticks="outside", ticklen=10, tickwidth=2,
                            showline=True, linewidth=2, linecolor="black", mirror=True)
        fig_png.update_yaxes(showticklabels=False, ticks="outside", ticklen=10, tickwidth=2,
                            showline=True, linewidth=2, linecolor="black", mirror=True)
        if show_legend:
            for tr in _family_legend_traces():
                fig_png.add_trace(tr)
            fig_png.update_layout(title=None, margin=dict(t=40, b=150),
                                  legend=dict(orientation="h", x=0.5, xanchor="center",
                                             y=-0.25, yanchor="top", font=dict(size=45)))
        else:
            fig_png.update_layout(title=None, margin=dict(t=40, b=40), showlegend=False)

        fig_png.write_image(str(OUT_DIR / f"{outstem}.png"))

    fig.show()
    return fig

pca_datasets = [
    (f"Landmarks (N={sparse_coords.shape[1]})", pca_sparse["x"],  pca_sparse["prop"]),
    (f"Dense correspondences (N={dense_coords.shape[1]})", pca_dense["x"], pca_dense["prop"]),
    ("NSM latents",           latent_scores,  latent_prop),
]

_ = pc_pair_grid(pca_datasets, 0, 1, "PC1 vs PC2 — sparse vs dense vs latents",
                 outstem=f"{RUN}_pca_1v2_comparison", show_legend=False)

In [ ]:
_ = pc_pair_grid(pca_datasets, 2, 3, "PC3 vs PC4 — sparse vs dense vs latents",
                 outstem=f"{RUN}_pca_3v4_comparison", show_legend=False)

### t-SNE

In [ ]:
# t-SNE on all three, side by side. Dense gets pre-reduced to 50 PCs first (sklearn's own
# recommendation, and the practical difference between seconds and tens of minutes at this width).

def tsne_scores(X, n_dim_cap=50):
    if X.shape[1] > n_dim_cap:
        X = X[:, :n_dim_cap]
    return TSNE(n_components=2, perplexity=30, learning_rate=50, early_exaggeration=12,
               n_iter_without_progress=2000, metric="cosine", random_state=42).fit_transform(X)

tsne_sparse = tsne_scores(pca_sparse["x"])
tsne_dense  = tsne_scores(pca_dense["x"])
tsne_latent = tsne_scores(codes)

tsne_datasets = [
    (f"Landmarks (N={sparse_coords.shape[1]})", tsne_sparse, None),
    (f"Dense correspondences (N={dense_coords.shape[1]})", tsne_dense,  None),
    ("NSM latents",           tsne_latent, None),
]
_ = pc_pair_grid(tsne_datasets, 0, 1, "t-SNE — sparse vs dense vs latents",
                 axis_prefix="t-SNE ", outstem=f"{RUN}_tsne_comparison", show_legend=False)

### UMAP

In [ ]:
# UMAP on all three, side by side (same 50-PC pre-reduction as t-SNE for the dense case)

def umap_scores(X, n_dim_cap=50):
    if X.shape[1] > n_dim_cap:
        X = X[:, :n_dim_cap]
    reducer = umap.UMAP(n_components=2, n_neighbors=50, min_dist=0.1, spread=0.5,
                        n_epochs=500, random_state=42)
    return reducer.fit_transform(X)

umap_sparse = umap_scores(pca_sparse["x"])
umap_dense  = umap_scores(pca_dense["x"])
umap_latent = umap_scores(codes)

umap_datasets = [
    (f"Landmarks (N={sparse_coords.shape[1]})", umap_sparse, None),
    (f"Dense correspondences (N={dense_coords.shape[1]})", umap_dense,  None),
    ("NSM latents",           umap_latent, None),
]
_ = pc_pair_grid(umap_datasets, 0, 1, "UMAP — sparse vs dense vs latents",
                 axis_prefix="UMAP ", outstem=f"{RUN}_umap_comparison", show_legend=True)

In [ ]:
# Stitch PCA / t-SNE / UMAP comparison PNGs into one 3x3 publication panel

from PIL import Image, ImageDraw, ImageFont

panel_files = [
    f"{RUN}_pca_1v2_comparison.png",
    f"{RUN}_tsne_comparison.png",
    f"{RUN}_umap_comparison.png",
]

try:
    font = ImageFont.truetype("Arial.ttf", 40)
except OSError:
    try:
        font = ImageFont.truetype("DejaVuSans.ttf", 40)   # non-bold, closer to Plotly's default weight
    except OSError:
        font = ImageFont.load_default()

imgs = [Image.open(OUT_DIR / fname) for fname in panel_files]
widths, heights = zip(*(im.size for im in imgs))
assert len(set(widths)) == 1, f"panel widths differ: {widths} -- regenerate with matching `width=` in pc_pair_grid"

panel_w = widths[0]
total_h = sum(heights)

COL_LABELS = ["Landmarks (N=28)", "Dense correspondences (N=4943)", "NSM latents"]
COL_HEADER_H = 70

grid = Image.new("RGB", (panel_w, COL_HEADER_H + total_h), "white")
draw = ImageDraw.Draw(grid)

col_w = panel_w // len(COL_LABELS)
for i, label in enumerate(COL_LABELS):
    bbox = draw.textbbox((0, 0), label, font=font)
    tx = i * col_w + (col_w - (bbox[2] - bbox[0])) / 2
    draw.text((tx, COL_HEADER_H - 45), label, fill="black", font=font)

y = COL_HEADER_H
for im in imgs:
    grid.paste(im, (0, y))
    y += im.height

outpath = OUT_DIR / f"{RUN}_pca_tsne_umap_3x3_panel.png"
grid.save(outpath, dpi=(300, 300))
print(f"Wrote {outpath.resolve()}  ({grid.size[0]}x{grid.size[1]} px)")
grid

## Export PC/tSNE/UMAP coords for R / Excel\n---

In [ ]:
# One tidy CSV per (dataset, method) combination -- named clearly rather than concatenated,
# so it is obvious in R/Excel which columns came from which representation.

def export_scores(name, scores, n_components=4):
    export = specimens[["specimen_id", "vertebra", "family", "genus", "species", "trait"]].copy()
    for i in range(min(n_components, scores.shape[1])):
        export[f"{name}{i+1}"] = scores[:, i]
    export = export.sort_values(["family", "genus", "species", "vertebra"])
    export.to_csv(OUT_DIR / f"{name.lower()}_points_for_stats.csv", index=False)
    return export

export_scores("PC_sparse",   pca_sparse["x"])
export_scores("PC_dense",    pca_dense["x"])
export_scores("PC_latent",   latent_scores)
export_scores("tSNE_sparse", tsne_sparse, n_components=2)
export_scores("tSNE_dense",  tsne_dense,  n_components=2)
export_scores("tSNE_latent", tsne_latent, n_components=2)
export_scores("UMAP_sparse", umap_sparse, n_components=2)
export_scores("UMAP_dense",  umap_dense,  n_components=2)
export_scores("UMAP_latent", umap_latent, n_components=2)

print(f"Wrote to {OUT_DIR.resolve()}:")
for f in sorted(OUT_DIR.glob("*_points_for_stats.csv")):
    print("  ", f.name)